In [1]:
!pip install quimb mps-to-circuit qiskit qiskit_aer

In [2]:
"""
main.py -- first-order Trotter sweep of the ground-state filter on a grid of n
equal steps (T = sum t_i = n * dt, every pulse t_i = k_i * dt).

Requires the cleaned builder.py (explicit-argument helpers) and your
get_energies / mps_to_circuit modules.
"""
import csv

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from qiskit.quantum_info import SparsePauliOp

from builder import (
    FilterBuilder, MPO_ham_j1j2, get_energies, get_spectrum, j1j2_hamiltonian,
    alpha_comm, apply_filter_pulse, build_rodeo_circuit, cx_depth_per_step,
    grid_filter, postselected_run, postselected_run_exact, resource_costs,
    state_metrics,
)
from mps_to_circuit import mps_to_circuit

# ----------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------
N = 8
J1, J2 = 1.0, 0.7
N_SWEEP = [10, 20, 40, 80, 160, 320]   # total Trotter steps n (dt = T / n)
SWEEP_L = 1                            # trial-circuit layers
VERIFY_COST_N = 10                     # full transpile at this n to check cx_per_step * n (None = skip)
ORDER = 1                              # first-order Trotter

# ----------------------------------------------------------------------
# Hamiltonian, reference ground states, spectrum
# ----------------------------------------------------------------------
H_mpo = MPO_ham_j1j2(N, j1=J1, j2=J2, cyclic=False)
res = get_energies(H_mpo, N=N, j1=J1, j2=J2, run_ed_check=True)
E0_dmrg, psi0_mps, ed = float(np.real(res["E0"])), res["psi0"], res["ed"]
if ed is None:
    raise RuntimeError("ED cross-check didn't run (check ED_CHECK_MAX_N / N).")
psi0_ed = ed["psi0_ed"]

psi0_dmrg = np.asarray(psi0_mps.to_dense()).reshape(-1)
psi0_dmrg = psi0_dmrg / np.linalg.norm(psi0_dmrg)

H_qk = j1j2_hamiltonian(N, j1=J1, j2=J2)
spec = get_spectrum(N=N, j1=J1, j2=J2, source="dmrg")
E0, Etop = spec["E0"], spec["Etop"]
H_scaled_qk = ((H_qk + SparsePauliOp.from_list([("I" * N, -E0)]))
               / (Etop - E0)).simplify()
Hm = H_scaled_qk.to_matrix()

# ----------------------------------------------------------------------
# Optimize the filter (times, phases); T = sum of times
# ----------------------------------------------------------------------
builder = FilterBuilder(total_time=spec["totaltime"], energies=spec["energies"])
best = builder.build(method="v3")[-1]
sw_times, sw_phases = best["times"], best["phases"]
T = float(np.sum(sw_times))

# ----------------------------------------------------------------------
# Trial circuit
# ----------------------------------------------------------------------
trial_qc = mps_to_circuit(psi0_mps.arrays, method="approximate",
                          shape="lpr", num_layers=SWEEP_L)
if trial_qc.num_qubits != N:
    raise ValueError(f"trial circuit has {trial_qc.num_qubits} qubits, expected {N}")

print(f"\nFilter: {len(sw_times)} optimized pulses, T = sum t_i = {T:.4f} "
      f"(pi/gap = {spec['totaltime']:.4f}), t_i in "
      f"[{sw_times.min():.3f}, {sw_times.max():.3f}] (scaled-H units)")

# ----------------------------------------------------------------------
# Exact references
# ----------------------------------------------------------------------
p_ex, sv_ex = postselected_run_exact(trial_qc, sw_times, sw_phases, Hm)
E_ex, Fd_ex, Fe_ex = state_metrics(sv_ex, H_qk, psi0_dmrg, psi0_ed)
print(f"[exact, optimizer times]  P_succ={p_ex:.6f}  E-E0dmrg={E_ex - E0_dmrg:.3e}  "
      f"F_dmrg={Fd_ex:.6f}  F_ed={Fe_ex:.6f}")

# ----------------------------------------------------------------------
# Cost of one Trotter step (transpiled once), and the commutator constant
# ----------------------------------------------------------------------
cx1, d1 = cx_depth_per_step(H_scaled_qk, order=ORDER)
alpha = alpha_comm(H_scaled_qk, tight=True)
print(f"alpha (nested commutators) = {alpha:.4f}   CX/step = {cx1}   depth/step = {d1}")

# ----------------------------------------------------------------------
# Sweep over n: snap to grid, re-optimize phases, Trotterize with k_i steps
# ----------------------------------------------------------------------
trot_rows = []
for n in N_SWEEP:
    k, dt, ph = grid_filter(sw_times, sw_phases, T, n, spec["energies"])
    tg = k * dt

    # exact filter on this grid: isolates the Trotter error from snapping loss
    p_g, sv_g = postselected_run_exact(trial_qc, tg, ph, Hm)
    _, Fd_g, Fe_g = state_metrics(sv_g, H_qk, psi0_dmrg, psi0_ed)

    p, sv = postselected_run(
        trial_qc, tg, ph,
        lambda qc, s, a, t, phi, i, k=k: apply_filter_pulse(
            qc, s, a, H_scaled_qk, t, phi, int(k[i]), order=ORDER))
    E, Fd, Fe = state_metrics(sv, H_qk, psi0_dmrg, psi0_ed)
    F_vs_grid = float(abs(np.vdot(sv_g, sv)) ** 2)

    row = {
        "n": n, "dt": dt, "n_pulses": len(k), "k": k.tolist(),
        "alpha": alpha, "bound_total": alpha * T ** 2 / (2 * n),
        "state_err": float(np.sqrt(2 * max(1 - np.sqrt(F_vs_grid), 0.0))),
        "P_succ_exact_grid": p_g, "P_succ": p,
        "F_dmrg_exact_grid": Fd_g, "F_ed_exact_grid": Fe_g,
        "F_dmrg": Fd, "F_ed": Fe, "E-E0_dmrg": E - E0_dmrg,
        "F_vs_exact_grid": F_vs_grid,
        "cx": cx1 * n, "depth": d1 * n,          # estimates
    }
    trot_rows.append(row)
    print(f"[n={n:>4}] dt={dt:.4f} k={k.tolist()}  P_succ={p:.6f} "
          f"(grid exact {p_g:.6f})  F_dmrg={Fd:.6f} (grid exact {Fd_g:.6f})  "
          f"1-F_vs_grid={1 - F_vs_grid:.2e}  bound={row['bound_total']:.2e}  "
          f"CX~{row['cx']}")

    if VERIFY_COST_N is not None and n == VERIFY_COST_N:
        qc_r, _ = build_rodeo_circuit(H_scaled_qk, tg, ph,
                                      trial_prep=lambda qc, s: None,
                                      trotter_steps=[int(x) for x in k],
                                      order=ORDER)
        c = resource_costs(qc_r, label=f"n={n} (full transpile)", flatten=True)
        print(f"  cost check at n={n}: measured CX={c['cx']}, depth={c['depth']}  "
              f"vs estimate CX={cx1 * n}, depth={d1 * n}")

# ----------------------------------------------------------------------
# Save
# ----------------------------------------------------------------------
with open("trotter_sweep.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(trot_rows[0].keys()))
    w.writeheader()
    w.writerows(trot_rows)

# ----------------------------------------------------------------------
# Plots
# ----------------------------------------------------------------------
ns = [r["n"] for r in trot_rows]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# 1) measured state error vs nested-commutator bound
ax[0].loglog(ns, [max(r["state_err"], 1e-16) for r in trot_rows], "o-",
             label=r"actual $\sqrt{2(1-\sqrt{F})}$")
ax[0].loglog(ns, [r["bound_total"] for r in trot_rows], "k--",
             label=r"bound $\alpha T^2/2n$")
ax[0].set_xlabel("total Trotter steps n (dt = T/n)")
ax[0].set_ylabel("state error vs exact grid filter")
ax[0].set_title("Trotter error vs bound")
ax[0].legend()

# 2) fidelity and success probability
ax[1].semilogx(ns, [r["F_dmrg"] for r in trot_rows], "o-", color="C0",
               label="F vs DMRG (Trotter)")
ax[1].semilogx(ns, [r["F_dmrg_exact_grid"] for r in trot_rows], "o:", color="C0",
               alpha=0.6, label="F vs DMRG (exact, grid)")
ax[1].set_xlabel("total Trotter steps n")
ax[1].set_ylabel("post-filter fidelity", color="C0")
axr = ax[1].twinx()
axr.semilogx(ns, [r["P_succ"] for r in trot_rows], "s-", color="C1",
             label="P_succ (Trotter)")
axr.semilogx(ns, [r["P_succ_exact_grid"] for r in trot_rows], "s:", color="C1",
             alpha=0.6, label="P_succ (exact, grid)")
axr.set_ylabel("success probability", color="C1")
ax[1].set_title("Fidelity and success prob (dotted = exact)")
l1, n1 = ax[1].get_legend_handles_labels()
l2, n2 = axr.get_legend_handles_labels()
ax[1].legend(l1 + l2, n1 + n2, loc="lower right")

# 3) cost (estimate: per-step count x n)
ax[2].loglog(ns, [r["cx"] for r in trot_rows], "o-", label="CX")
ax[2].loglog(ns, [r["depth"] for r in trot_rows], "s-", label="depth")
ax[2].set_xlabel("total Trotter steps n")
ax[2].set_ylabel("count (basis h,s,cx,rz)")
ax[2].set_title("Filter cost (estimate)")
ax[2].legend()

fig.suptitle(f"First-order Trotter, grid n sweep, N={N}, L={SWEEP_L}, J2/J1 = {J2 / J1}")
fig.tight_layout()
fig.savefig("NeWtrotter_sweep.png", dpi=150)
plt.close(fig)
print("saved trotter_sweep.csv, trotter_sweep.png")

Running DMRG for ground state...
Running DMRG for highest excited state (via -H)...
  spectral bandwidth (E_top - E0) = 5.92474-5.55112e-16j, PENALTY = 6.51721-6.10623e-16j
Building orthogonal-projector penalty MPO from ground state...
Running DMRG for first excited state (PENALTY = 6.51721-6.10623e-16j)...
Running ED cross-check (N=8, dim=2^8=256)...
  [OK] E0: DMRG=-3.1247352417-0.0000000000j  ED=-3.1247352417  |diff|=5.77e-15
  [OK] E1: DMRG=-2.4825050114  ED=-2.4825050114  |diff|=3.36e-12
  [OK] E_top: DMRG=2.8000000000-0.0000000000j  ED=2.8000000000  |diff|=1.08e-15
[source=dmrg] DMRG spectrum (E0, E1, E_top only), N=8
Running DMRG for ground state...
Running DMRG for highest excited state (via -H)...
  spectral bandwidth (E_top - E0) = 5.92474+9.99201e-16j, PENALTY = 6.51721+1.09912e-15j
Building orthogonal-projector penalty MPO from ground state...
Running DMRG for first excited state (PENALTY = 6.51721+1.09912e-15j)...
Running ED cross-check (N=8, dim=2^8=256)...
  [OK] E0: DMR